# 09회차 실습 노트북 — 회귀모형(1)

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adminmrobo/fintech_data_analysis/blob/main/notebooks/09_regression.ipynb)

해설 페이지: [https://adminmrobo.github.io/fintech_data_analysis/09_regression.html](https://adminmrobo.github.io/fintech_data_analysis/09_regression.html)

위에서부터 셀을 차례로 실행하세요(Shift+Enter). 데이터는 저장소 `data/` 폴더(fintech-data 복사본)에서 읽으며, 무작위 시드를 42로 고정해 해설 페이지의 대결 숫자와 같은 값이 나옵니다.

글 · 안상선 ((주)M-Robo 대표)

In [1]:
# 준비: 라이브러리와 데이터 불러오기 (Colab에서는 그대로 실행하면 됩니다)
import os, urllib.parse, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)

BASE = 'https://raw.githubusercontent.com/adminmrobo/fintech_data_analysis/main/data/'
def load(path, **kw):
    """저장소의 data/ 폴더에서 CSV를 읽는다. 내 컴퓨터에 저장소가 있으면 그 파일을, 없으면 GitHub에서 읽는다."""
    for root in ('data', '../data'):
        f = os.path.join(root, path)
        if os.path.exists(f):
            return pd.read_csv(f, **kw)
    return pd.read_csv(BASE + urllib.parse.quote(path), **kw)

RS = 42   # 해설 페이지와 같은 숫자가 나오도록 무작위 시드를 고정

## 1. 보험료를 나이·BMI·흡연으로 예측

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error
ins = load('Revolution/insurance.csv'); ins['sm'] = (ins.smoker == 'yes').astype(int)
tr, te = train_test_split(ins, test_size=.25, random_state=RS)
for cols in (['age'], ['age', 'bmi'], ['age', 'bmi', 'sm']):
    m = LinearRegression().fit(tr[cols], tr.charges)
    print(cols, 'R² =', round(r2_score(te.charges, m.predict(te[cols])), 3), ' 계수', dict(zip(cols, m.coef_.round(0).tolist())))

['age'] R² = 0.099  계수 {'age': 252.0}
['age', 'bmi'] R² = 0.129  계수 {'age': 236.0, 'bmi': 331.0}
['age', 'bmi', 'sm'] R² = 0.762  계수 {'age': 262.0, 'bmi': 327.0, 'sm': 23668.0}


## 2. 대결 재현: 데이터 범위 밖(외삽) — 선형 vs KNN 회귀
쇼핑몰 고객 중 연봉 하위 90%로 학습하고 상위 10% 고객의 소비액을 맞혀 봅니다.

In [3]:
dm = load('Revolution/DirectMarketing.csv')
x = dm.Salary.values.reshape(-1, 1) / 1000; y = dm.AmountSpent.values
cut = np.quantile(x[:, 0], .9); lo, hi = x[:, 0] <= cut, x[:, 0] > cut
knn = KNeighborsRegressor(5).fit(x[lo], y[lo]); lin = LinearRegression().fit(x[lo], y[lo])
print(f'경계 연봉 {cut:.1f}천 달러, 상위 고객 {hi.sum()}명, 실제 평균 소비 {y[hi].mean():.0f}')
print('KNN MAE', round(mean_absolute_error(y[hi], knn.predict(x[hi]))), '  KNN 예측 최댓값', knn.predict(x[hi]).max())
print('선형 MAE', round(mean_absolute_error(y[hi], lin.predict(x[hi]))))

경계 연봉 99.2천 달러, 상위 고객 100명, 실제 평균 소비 2417
KNN MAE 1351   KNN 예측 최댓값 3352.8
선형 MAE 944


In [4]:
xs = np.linspace(x.min(), x.max(), 300).reshape(-1, 1)
plt.scatter(x, y, s=4, c=np.where(hi, 'tab:red', 'tab:gray'))
plt.plot(xs, knn.predict(xs), label='KNN'); plt.plot(xs, lin.predict(xs), label='linear')
plt.axvline(cut, ls='--', c='k'); plt.xlabel('salary (k$)'); plt.ylabel('amount spent'); plt.legend(); plt.show()

**AI 판단 연습** — KNN 예측이 경계선 오른쪽에서 평평해지는 이유를 "이웃"이라는 말로 설명해 보세요.